# Project 7: Incident response

## 1. What can happen?

A privacy incident may require stopping an assistant until recovery requirements are met.

**Example** An incident is marked resolved, but the required retest has not been linked to its recovery record.

## 2. Our idea

We use fictional incidents, recovery evidence, and sample reviewer authority. A local Python responder records actual calls in a list. The status check runs before that responder. Sample approvals are fictional; your own recovery record stays pending.

### How we do it

1. Inspect the confirmed incident and the unconfirmed report.
2. Show that the missing retest keeps the responder suspended, even though the incident says resolved.
3. Link one existing synthetic retest and compare the same request. Inspect your pending recovery template.

## 3. How it helps

The blocked request makes zero responder calls. Linking the matching passed retest completes this fictional recovery case and permits one local call. A real recovery also needs trustworthy investigation, retests, and an authenticated decision.

## 4. Input and setup

Run uv sync --locked at the outer root. Open this numbered folder in VS Code,
select the root .venv/Scripts/python.exe kernel, and run one cell at a time.
All source examples are fictional and make no model calls.


### Step 1: Check the lesson folder

This cell only checks the working directory and imports standard-library tools.

In [ ]:
import json
from copy import deepcopy
from datetime import date
from pathlib import Path

project_dir = Path.cwd()
assert project_dir.name == "7_incident_response", "Open this numbered folder in VS Code."
as_of = date(2026, 11, 15)
print("Lesson:", project_dir.name, "| fixed sample date:", as_of)

### Step 2: Read incident and recovery fixtures

The evidence and approvals are synthetic examples, not events or judgments performed by this notebook.

In [ ]:
def read(name):
    return json.loads((project_dir / "docs/dataset" / name).read_text())

release, incidents, policy = read("release.json"), read("incidents.json"), read("policy.json")
reviewers, evidence, recoveries = read("reviewers.json"), read("evidence.json"), read("recoveries.json")
question = "How many annual leave days are in the policy?"
print(json.dumps({"incidents": incidents, "recoveries": recoveries}, indent=2))

### Step 3: Enforce the suspension

Read tools/incidents.py and models/demo.py. application_state requires containment, a matching passed retest, and an authorized recovery approval for each confirmed incident. guarded_reply checks that state before calling respond.

In [ ]:
from tools.incidents import application_state, guarded_reply

baseline_state = application_state(incidents, release, recoveries, evidence, reviewers, policy)
baseline_calls = []
baseline = guarded_reply(question, baseline_state, baseline_calls)
print(baseline_state)
print(baseline, "| actual local calls:", len(baseline_calls))

### Step 4: Inspect the unconfirmed edge case

This policy allows the assistant to remain active while an unconfirmed report awaits investigation. That is an example policy choice, not a general rule for every organization.

In [ ]:
unconfirmed = [item for item in incidents if item["confirmation"] == "unconfirmed"]
normal_state = application_state(unconfirmed, release, recoveries, evidence, reviewers, policy)
normal_calls = []
print(guarded_reply(question, normal_state, normal_calls))
print("Actual local calls:", len(normal_calls))

### Step 5: Link one existing retest fixture

Change only the retest reference. The containment and fictional authorized approval were already present. We create no new human decision.

In [ ]:
candidate_recoveries = deepcopy(recoveries)
candidate_recoveries["INC-001"]["retest_ref"] = "RETEST-001"
candidate_state = application_state(incidents, release, candidate_recoveries, evidence, reviewers, policy)
print(candidate_state)

### Step 6: Try the identical request again

The call log demonstrates actual enforcement rather than only printing a status label.

In [ ]:
candidate_calls = []
candidate = guarded_reply(question, candidate_state, candidate_calls)
print(candidate)
print("Actual local responder calls:", len(baseline_calls), "->", len(candidate_calls))
print("Real model/API calls:", candidate["model_api_calls"])

### Step 7: Read your own pending recovery record

The learner record is separate from the approved fixture. The notebook does not write investigation, retest, or approval decisions for you.

In [ ]:
learner_recovery = read("learner_recovery.json")
print(json.dumps(learner_recovery, indent=2))

### Explain the result

An unconfirmed-only case remains active under this example policy. The confirmed case is suspended until the retest is linked: actual local responder calls change from 0 to 1. Status resolved alone does not reopen the assistant. Every confirmed in-scope incident needs its own recovery.

The blocked request makes zero responder calls. Linking the matching passed retest completes this fictional recovery case and permits one local call. A real recovery also needs trustworthy investigation, retests, and an authenticated decision.

Changes are in memory. This notebook writes no files.

Pause here. Your run and any human decisions remain your learning steps.